# Étape 2 — Chargement et unification des données
**Projet : Les LLM parlent-ils maghrébin ?** · Rôle Données

Ce notebook charge chaque dataset brut et le convertit dans **un format unique** :

| colonne | contenu |
|---|---|
| `id` | identifiant unique `dataset:split:numéro` |
| `text` | le texte brut (pas encore nettoyé : ce sera l'étape 3) |
| `dialect` | `DZ`, `MA`, `TN`, `MSA`, `FR`, `EN` |
| `script` | `arab`, `latin`, `mixed`, `other` (détecté automatiquement) |
| `task` | `sentiment`, `fakenews`, `lm` (pré-entraînement), `parallel`, `did` |
| `label` | étiquette harmonisée (`neg`/`neu`/`pos` pour le sentiment) |
| `label_orig` | étiquette d'origine, pour pouvoir revenir en arrière |
| `dataset` | nom du dataset |
| `sub_source` | sous-source quand elle existe (ex. youtube / translated pour FASSILA) |
| `orig_split` | split d'origine (`train`, `dev`, `test`, `all`) |
| `group_id` | pour les corpus parallèles : même valeur = même phrase traduite (évite la fuite train/test) |
| `usage` | `train_ok` ou `eval_only` (FLORES+ ne doit jamais servir à l'entraînement) |

**À la fin, copie le tableau récapitulatif et colle-le à Claude** pour passer à l'étape 3 (nettoyage).

## 0. Installation et connexion

In [ ]:
!pip -q install datasets pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Connexion Hugging Face (nécessaire pour FLORES+ et Atlaset, datasets protégés).
# Crée un token "Read" sur https://huggingface.co/settings/tokens puis colle-le quand on te le demande.
from huggingface_hub import login
login()

## 1. Chemins

Range les fichiers téléchargés comme ceci dans ton Drive (dézippe les ZIP) :
```
MyDrive/darija_project/data/raw/
    algerian_dialect/   ← le contenu du ZIP Mendeley (le CSV ADArabic-...-45000.csv)
    fassila/            ← le dossier FASSILA-main dézippé
    tsac/               ← le dossier TSAC-master dézippé
    tunizi/             ← le(s) fichier(s) Zenodo (tunizi_train ...)
    madar/              ← plus tard, quand le lien MADAR arrive par mail
```

In [ ]:
from pathlib import Path
import re, glob, unicodedata
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/darija_project')   # ← adapte si besoin
RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

for d in ['algerian_dialect', 'fassila', 'tsac', 'tunizi', 'madar']:
    p = RAW / d
    n = len([f for f in p.rglob('*') if f.is_file()]) if p.exists() else 0
    print(f"{d:18s} {'OK' if n else 'MANQUANT'}  ({n} fichiers)")

## 2. Outils communs

`detect_script` compte les lettres arabes et latines. Un texte en `latin` peut être de l'**arabizi**
(« 3lach », « 7aja ») **ou du français** : on les séparera à l'étape 3.

In [ ]:
AR = re.compile(r'[؀-ۿݐ-ݿࢠ-ࣿﭐ-﷿ﹰ-﻿]')
LAT = re.compile(r'[A-Za-zÀ-ÿ]')

def detect_script(text: str) -> str:
    if not isinstance(text, str):
        return 'other'
    a, l = len(AR.findall(text)), len(LAT.findall(text))
    if a + l == 0:
        return 'other'
    r = a / (a + l)
    if r >= 0.8:
        return 'arab'
    if r <= 0.2:
        return 'latin'
    return 'mixed'

COLS = ['id', 'text', 'dialect', 'script', 'task', 'label', 'label_orig',
        'dataset', 'sub_source', 'orig_split', 'group_id', 'usage']

def finalize(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Complète les colonnes manquantes, crée les id, détecte l'écriture."""
    df = df.copy()
    df['dataset'] = dataset
    for c in COLS:
        if c not in df.columns:
            df[c] = None
    df['usage'] = df['usage'].fillna('train_ok')
    df['text'] = df['text'].astype(str).str.strip()
    for c in ['label', 'label_orig', 'sub_source', 'group_id', 'orig_split']:
        df[c] = df[c].map(lambda x: None if x is None or (isinstance(x, float) and np.isnan(x)) else str(x))
    df = df[df['text'].str.len() > 0]
    df['id'] = [f"{dataset}:{s}:{i}" for i, s in enumerate(df['orig_split'].fillna('all'))]
    df['script'] = df['text'].map(detect_script)
    return df[COLS].reset_index(drop=True)

def best_text_column(df: pd.DataFrame, n=300) -> str:
    """Pour un dataset inconnu : la colonne texte qui contient le plus de lettres arabes."""
    scores = {}
    for c in df.columns:
        s = df[c].head(n)
        if s.map(lambda x: isinstance(x, str)).mean() < 0.9:
            continue
        joined = ' '.join(s)
        scores[c] = len(AR.findall(joined)) / max(len(joined), 1)
    return max(scores, key=scores.get) if scores else None

ALL = {}   # nom du dataset -> DataFrame unifié

## 3. Algerian Dialect (Mendeley) — DZ, sentiment 5 niveaux
Colonnes d'origine : 1 = commentaire, 2 = label 0 à 4 (très négatif → très positif), puis métadonnées.
Harmonisation : 0-1 → `neg`, 2 → `neu`, 3-4 → `pos`. On garde `label_orig` pour l'analyse fine.

In [ ]:
f = sorted((RAW / 'algerian_dialect').rglob('*.csv'))
print(f)
raw = pd.read_csv(f[0])
print(raw.shape, list(raw.columns)); display(raw.head(3))

ad = pd.DataFrame({
    'text': raw.iloc[:, 0],
    'label_orig': raw.iloc[:, 1],
})
ad['label'] = ad['label_orig'].map({0: 'neg', 1: 'neg', 2: 'neu', 3: 'pos', 4: 'pos'})
ad['dialect'], ad['task'], ad['orig_split'], ad['sub_source'] = 'DZ', 'sentiment', 'all', 'youtube'
ALL['algerian_dialect'] = finalize(ad, 'algerian_dialect')
print(ALL['algerian_dialect'].label.value_counts(dropna=False))

## 4. FASSILA — DZ, sentiment (3 classes) + fake news
Vérifié dans le dépôt : sentiment `0 = négatif`, `1 = positif`, `2 = neutre`.

⚠️ **Attention** : la majorité de FASSILA est **traduite ou paraphrasée depuis des dépêches en arabe standard**
(colonne `source` : translated / paraphrased / manual) ; seuls ~900 textes par split viennent de YouTube.
On garde cette info dans `sub_source`. Les deux tâches partagent beaucoup de phrases : on dédupliquera à l'étape 3.

In [ ]:
fas_dir = next((RAW / 'fassila').rglob('Data'))
parts = []
for task_dir, task in [('Sentiment Analysis', 'sentiment'), ('Fake News', 'fakenews')]:
    for split in ['train', 'validation', 'test']:
        d = pd.read_csv(fas_dir / task_dir / f'{split}_set.csv')
        d = d.rename(columns={'source': 'sub_source', 'label': 'label_orig'})
        d['task'] = task
        d['orig_split'] = {'validation': 'dev'}.get(split, split)
        if task == 'sentiment':
            d['label'] = d['label_orig'].map({0: 'neg', 1: 'pos', 2: 'neu'})
        else:
            d['label'] = 'fn_' + d['label_orig'].astype(str)   # sens de 0/1 à vérifier dans l'article
        parts.append(d)
fas = pd.concat(parts, ignore_index=True)
fas['sub_source'] = fas['sub_source'].astype(str).str.strip()
fas['dialect'] = 'DZ'
ALL['fassila'] = finalize(fas, 'fassila')
print(pd.crosstab(ALL['fassila'].task, ALL['fassila'].sub_source))

## 5. TSAC — TN, sentiment binaire
4 fichiers texte, une phrase par ligne. **Mélange d'écritures** (arabe, arabizi, un peu de français) :
la colonne `script` permettra de les séparer.

In [ ]:
tsac_dir = next((RAW / 'tsac').rglob('train_pos.txt')).parent
parts = []
for split in ['train', 'test']:
    for pol in ['pos', 'neg']:
        lines = (tsac_dir / f'{split}_{pol}.txt').read_text(encoding='utf-8').splitlines()
        parts.append(pd.DataFrame({'text': lines, 'label': pol, 'label_orig': pol, 'orig_split': split}))
tsac = pd.concat(parts, ignore_index=True)
tsac['dialect'], tsac['task'], tsac['sub_source'] = 'TN', 'sentiment', 'facebook'
ALL['tsac'] = finalize(tsac, 'tsac')
print(pd.crosstab(ALL['tsac'].orig_split, ALL['tsac'].script))

## 6. TUNIZI — TN, arabizi, sentiment
Le format du fichier Zenodo n'est pas documenté : on l'inspecte d'abord.
La version GitHub (3 000 lignes) utilise le format `label;texte` avec `1` = positif et `-1` = négatif.

In [ ]:
tun_files = [f for f in (RAW / 'tunizi').rglob('*') if f.is_file()]
for f in tun_files:
    print('====', f.name, f.stat().st_size // 1024, 'Ko')
    print(open(f, encoding='utf-8', errors='replace').read(400))

In [ ]:
def read_tunizi(path):
    first = open(path, encoding='utf-8', errors='replace').readline()
    # 1) format GitHub "label;texte" (ex. "1;ana fil ghorba ...")
    if re.match(r'^\W?-?\d+;', first):
        rows = [l.rstrip('\n').split(';', 1) for l in open(path, encoding='utf-8') if ';' in l]
        return pd.DataFrame({'label_orig': [re.sub(r'[^\d-]', '', r[0]) for r in rows],
                             'text': [r[1] for r in rows]})
    # 2) sinon : CSV avec détection automatique du séparateur
    try:
        d = pd.read_csv(path, sep=None, engine='python')
        if d.shape[1] >= 2:
            text_col = best_text_column(d) or max(
                (c for c in d.columns if d[c].dtype == object),
                key=lambda c: d[c].astype(str).str.len().mean())
            label_col = next((c for c in d.columns if c.lower() in ('label', 'target', 'sentiment', 'polarity')), None)
            return pd.DataFrame({'text': d[text_col], 'label_orig': d[label_col] if label_col else None})
    except Exception as e:
        print('CSV impossible :', e)
    raise ValueError(f"Format inconnu pour {path.name} : colle les premières lignes à Claude.")

parts = []
for f in tun_files:
    d = read_tunizi(f)
    name = f.stem.lower()
    d['orig_split'] = 'train' if 'train' in name else 'dev' if ('dev' in name or 'val' in name) else 'test' if 'test' in name else 'all'
    parts.append(d)
tun = pd.concat(parts, ignore_index=True)
print('Valeurs de label trouvées :', tun['label_orig'].value_counts(dropna=False).head(10).to_dict())

LAB = {'1': 'pos', '-1': 'neg', '0': 'neu', 'positive': 'pos', 'negative': 'neg', 'neutral': 'neu'}
tun['label'] = tun['label_orig'].astype(str).str.strip().str.lower().map(LAB)
tun['dialect'], tun['task'], tun['sub_source'] = 'TN', 'sentiment', 'social'
ALL['tunizi'] = finalize(tun, 'tunizi')
print(pd.crosstab(ALL['tunizi'].orig_split, ALL['tunizi'].label, dropna=False))

## 7. FLORES+ — phrases parallèles, **évaluation uniquement**
Conditions acceptées sur Hugging Face : ne pas utiliser pour l'entraînement → `usage = eval_only`.
`group_id` = identifiant de la phrase : les 5 langues d'une même phrase ont le même `group_id`.
Pas d'algérien dans FLORES+ (c'est normal, on le compensera avec MADAR).

In [ ]:
from datasets import load_dataset

FLORES = {'fra_Latn': 'FR', 'arb_Arab': 'MSA', 'ary_Arab': 'MA', 'aeb_Arab': 'TN', 'eng_Latn': 'EN'}
parts = []
for cfg, dia in FLORES.items():
    for split in ['dev', 'devtest']:
        ds = load_dataset('openlanguagedata/flores_plus', cfg, split=split)
        d = ds.to_pandas()[['id', 'text']].rename(columns={'id': 'group_id'})
        d['group_id'] = 'flores_' + split + '_' + d['group_id'].astype(str)
        d['dialect'], d['orig_split'] = dia, split
        parts.append(d)
fl = pd.concat(parts, ignore_index=True)
fl['task'], fl['usage'] = 'parallel', 'eval_only'
ALL['flores_plus'] = finalize(fl, 'flores_plus')
print(pd.crosstab(ALL['flores_plus'].dialect, ALL['flores_plus'].orig_split))

## 8. DarijaBridge — MA
Les noms de colonnes ne sont pas documentés : on affiche la structure, puis on choisit automatiquement
la colonne qui contient le plus d'arabe. **Vérifie l'affichage** : si la colonne choisie n'est pas la darija, corrige `DB_COL`.

In [ ]:
db = load_dataset('M-A-D/DarijaBridge', split='train').to_pandas()
print(db.shape); print(db.dtypes); display(db.head(3))
DB_COL = best_text_column(db)
print('Colonne darija choisie :', DB_COL)

In [ ]:
dbu = pd.DataFrame({'text': db[DB_COL]})
dbu['dialect'], dbu['task'], dbu['orig_split'] = 'MA', 'lm', 'all'
ALL['darijabridge'] = finalize(dbu, 'darijabridge')
print(ALL['darijabridge'].script.value_counts())

## 9. Atlaset — MA, grand corpus (≈ 1,2 M lignes, 1,1 Go : compte 3-5 min)

⚠️ Atlaset mélange du texte authentique (réseaux sociaux, presse, sous-titres…) et des données
**synthétiques ou probablement traduites automatiquement** (jeux d'instructions anglais : sharegpt, flan, alpaca…).
On ne supprime rien ici : on marque ces sources dans `sub_source` et on décidera à l'étape 3, après avoir lu des exemples.

In [ ]:
at = load_dataset('atlasia/Atlaset')
print(at)
at_df = pd.concat([at[s].to_pandas().assign(orig_split=s) for s in at.keys()], ignore_index=True)
print(at_df.columns.tolist()); display(at_df.head(3))

In [ ]:
AT_TEXT = 'text' if 'text' in at_df.columns else best_text_column(at_df)
AT_SRC = next((c for c in at_df.columns if c.lower() in ('source', 'dataset', 'origin', 'src')), None)
print('texte :', AT_TEXT, '| source :', AT_SRC)

atu = pd.DataFrame({'text': at_df[AT_TEXT], 'orig_split': at_df['orig_split'].replace({'validation': 'dev'})})
atu['sub_source'] = at_df[AT_SRC].astype(str) if AT_SRC else None
atu['dialect'], atu['task'] = 'MA', 'lm'
ALL['atlaset'] = finalize(atu, 'atlaset')
if AT_SRC:
    print(ALL['atlaset'].sub_source.value_counts().head(50))

## 10. MADAR — à lancer quand le lien est arrivé par mail
Format attendu (corpus parallèle CAMeL) : fichiers TSV avec les colonnes `sentID.BTEC`, `split`, `lang`, `sent`.
On garde 6 « langues » : Alger, Rabat, Fès, Tunis, Sfax, MSA. Le `group_id` = `sentID.BTEC` servira au découpage
**par phrase** (sinon la même phrase serait en train dans un dialecte et en test dans un autre).

In [ ]:
madar_files = sorted((RAW / 'madar').rglob('*.tsv'))
print(len(madar_files), 'fichiers TSV')
if madar_files:
    m = pd.concat([pd.read_csv(f, sep='\t', quoting=3) for f in madar_files], ignore_index=True)
    print(m.columns.tolist()); print(m['lang'].value_counts().head(30) if 'lang' in m.columns else m.head())

In [ ]:
CITY = {'ALG': ('DZ', 'Alger'), 'RAB': ('MA', 'Rabat'), 'FES': ('MA', 'Fès'),
        'TUN': ('TN', 'Tunis'), 'SFX': ('TN', 'Sfax'), 'MSA': ('MSA', 'MSA')}
if madar_files:
    mm = m[m['lang'].isin(CITY)].copy()
    mm = mm.drop_duplicates(subset=['sentID.BTEC', 'lang', 'sent'])
    mm['text'] = mm['sent']
    mm['dialect'] = mm['lang'].map(lambda x: CITY[x][0])
    mm['sub_source'] = mm['lang'].map(lambda x: CITY[x][1])
    mm['group_id'] = 'madar_' + mm['sentID.BTEC'].astype(str)
    mm['orig_split'] = mm['split'].str.lower() if 'split' in mm.columns else 'all'
    mm['task'] = 'parallel'
    ALL['madar'] = finalize(mm, 'madar')
    print(pd.crosstab(ALL['madar'].sub_source, ALL['madar'].orig_split))
else:
    print('MADAR pas encore disponible : on passe.')

## 11. Fusion, sauvegarde et récapitulatif

In [ ]:
U = pd.concat(ALL.values(), ignore_index=True)
U['n_chars'] = U['text'].str.len()
U['n_words'] = U['text'].str.split().str.len()
U.to_parquet(INTERIM / 'unified_v0.parquet', index=False)
print('Sauvegardé :', INTERIM / 'unified_v0.parquet', U.shape)

In [ ]:
recap = (U.groupby(['dataset', 'dialect', 'task', 'usage'])
           .agg(lignes=('text', 'size'),
                mots=('n_words', 'sum'),
                mots_par_ligne=('n_words', 'median'),
                pct_arab=('script', lambda s: round(100 * (s == 'arab').mean(), 1)),
                pct_latin=('script', lambda s: round(100 * (s == 'latin').mean(), 1)),
                pct_mixed=('script', lambda s: round(100 * (s == 'mixed').mean(), 1)),
                doublons_exacts=('text', lambda s: int(s.duplicated().sum())))
           .reset_index())
print(recap.to_markdown(index=False))

In [ ]:
print('Mots par dialecte (hors eval_only) :')
print(U[U.usage == 'train_ok'].groupby('dialect')['n_words'].sum().sort_values(ascending=False).to_markdown())
print('\nSentiment, labels harmonisés :')
print(pd.crosstab([U[U.task == 'sentiment'].dataset], U[U.task == 'sentiment'].label, dropna=False).to_markdown())

### ✅ Fin de l'étape 2
Copie **les trois tableaux ci-dessus** (et les éventuelles erreurs) et colle-les à Claude.
Étape suivante : nettoyage (normalisation de l'arabe, séparation arabizi / français, déduplication,
suppression des pseudos et numéros) puis découpage train / val / test.